# HR Request Routing Study — Official Colab Run

This notebook compares a uniform Qwen2.5-3B baseline with a tiered rules → Qwen → human-review system. Use a **T4 GPU**, run the cells in order, and do not edit the test labels or routing rules after beginning the official test run.

## 1. Select the GPU
In Colab choose **Runtime → Change runtime type → T4 GPU**, then continue.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/MarieBelle88/HR-request-routing.git'
REPO_DIR = '/content/hr-request-routing-study'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import torch, platform
assert torch.cuda.is_available(), 'No GPU detected. Change the Colab runtime to T4 GPU.'
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

## 2. Verify the repository and inspect the development data
These checks do not produce paper results.

In [ ]:
!python scripts/verify_dataset.py
!python -m pytest -q

In [ ]:
import json, pandas as pd
records = [json.loads(line) for line in open('data/hr_requests.jsonl', encoding='utf-8')]
dev = pd.DataFrame([r for r in records if r['split'] == 'dev'])
display(dev[['request_id', 'message', 'gold_category', 'gold_urgency', 'gold_action', 'gold_ideal_route']])

## 3. Development run
Run this once to confirm that the model and output parser work. The model download may take several minutes.

In [ ]:
!python scripts/run_experiment.py --split dev --system both --output-dir results/dev
!python scripts/evaluate_results.py --baseline results/dev/dev_baseline.jsonl --tiered results/dev/dev_tiered.jsonl --output-dir results/dev

## 4. Locked official test run
Do not change prompts, labels, or rules after starting this cell. These are the results used in the paper.

In [ ]:
!python scripts/run_experiment.py --split test --system both --output-dir results
!python scripts/evaluate_results.py --baseline results/test_baseline.jsonl --tiered results/test_tiered.jsonl --output-dir results

In [ ]:
summary = json.load(open('results/experiment_summary.json', encoding='utf-8'))
display(pd.DataFrame({
    'Uniform Qwen': summary['baseline'],
    'Tiered system': summary['tiered']
}).loc[['category_accuracy','urgency_accuracy','action_accuracy','exact_resolution_accuracy','escalation_precision','escalation_recall','escalation_f1','qwen_calls','total_latency_seconds']])
print('Qwen-call reduction:', f"{summary['comparison']['qwen_call_reduction']:.1%}")
print('Latency reduction:', f"{summary['comparison']['latency_reduction']:.1%}")

## 5. Download the genuine results
Send the downloaded ZIP back in ChatGPT so the measured results can be inserted into the paper.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/HR_Routing_Official_Results', 'zip', root_dir=REPO_DIR, base_dir='results')
print('Created:', archive)
files.download(archive)